In [ ]:
import sys
import os
import random
import pandas as pd
import librosa
import torch
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import cv2
from torchaudio.transforms import Spectrogram
from typing import TypedDict
from collections.abc import Callable

from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_DIR))
print(PROJECT_DIR)

In [ ]:
from src.core.config import settings, update_path_settings
from src.domain.pipelines.annotations import load_annotation_file, get_annotation_file

update_path_settings(project_dir=PROJECT_DIR)
SECRETS_DIR = settings.SECRETS_DIR or Path("secrets")
DATA_ZIP_DIR = settings.DATA_ZIP_DIR or Path("data") / "zip"
DATA_RAW_DIR = settings.DATA_RAW_DIR or Path("data") / "raw"
DATA_PREPROCESSED_DIR = settings.DATA_PREPROCESSED_DIR or Path("data") / "preprocessed"

In [ ]:
random_dir = random.choice(
    [d for d in os.listdir(DATA_RAW_DIR) if (DATA_RAW_DIR / d).is_dir()]
)
SAMPLE_DIR = DATA_RAW_DIR / random_dir
ANNOTATIONS_DIR = DATA_PREPROCESSED_DIR / random_dir
SAMPLE_DIR, ANNOTATIONS_DIR

In [ ]:
records_files: list[Path] = [
    SAMPLE_DIR / wav_file
    for wav_file in os.listdir(SAMPLE_DIR)
    if wav_file.endswith(".wav")
]
sample_record: Path = random.choice(records_files)
sample_annotation: Path | None = get_annotation_file(ANNOTATIONS_DIR, sample_record)

annotations: pd.DataFrame = (
    load_annotation_file(sample_annotation) if sample_annotation else pd.DataFrame()
)

In [ ]:
class BBoxAnnotation(TypedDict):
    specie: str
    call_type: str
    begin_time: float
    end_time: float
    low_freq: float
    high_freq: float


class YoloCoordinates(TypedDict):
    class_id: int
    xc_rel: float
    yc_rel: float
    w_rel: float
    h_rel: float

In [ ]:
def audio_to_rgb_spectrogram(
    audio_clip: np.ndarray, nfft: int = 2048, hop_length: int = 512
) -> np.ndarray:
    spec_transform = Spectrogram(n_fft=nfft, hop_length=hop_length)
    spec_tensor = spec_transform(torch.from_numpy(audio_clip))

    db_spec = 10 * np.log10(spec_tensor.numpy() + 1e-10)
    db_spec = np.asarray(db_spec, dtype=np.float32)

    min_val = float(np.min(db_spec))
    max_val = float(np.max(db_spec))
    if max_val > min_val:
        norm_spec = (db_spec - min_val) * (255.0 / (max_val - min_val))
    else:
        norm_spec = np.zeros_like(db_spec, dtype=np.float32)

    norm_spec = np.clip(norm_spec, 0, 255).astype(np.uint8)
    norm_spec = cv2.flip(norm_spec, 0)

    rgb_image = cv2.applyColorMap(norm_spec, cv2.COLORMAP_VIRIDIS)

    return rgb_image


def annotations_to_yolo_coords(
    annotations: list[BBoxAnnotation],
    clip_duration: float,
    sample_rate: int,
    class_mapping: Callable[[str, str], int],
) -> list[YoloCoordinates]:
    nyquist_freq = sample_rate / 2.0
    yolo_coords = []

    for ann in annotations:
        class_id = class_mapping(ann["specie"], ann["call_type"])
        if class_id == -1:
            continue

        t_start = ann["begin_time"]
        t_end = ann["end_time"]
        xc_rel = ((t_start + t_end) / 2.0) / clip_duration
        w_rel = (t_end - t_start) / clip_duration

        f_low = ann["low_freq"]
        f_high = ann["high_freq"]

        f_center = (f_low + f_high) / 2.0
        yc_rel = 1.0 - (f_center / nyquist_freq)
        h_rel = (f_high - f_low) / nyquist_freq

        yolo_coord = YoloCoordinates(
            class_id=class_id, xc_rel=xc_rel, yc_rel=yc_rel, w_rel=w_rel, h_rel=h_rel
        )
        yolo_coords.append(yolo_coord)

    return yolo_coords

In [ ]:
from src.domain.pipelines.audio import extract_anchored_clip, extract_clip
from src.domain.utils.species import get_species_id

SAMPLE_RATES = [24000, 32000, 40000, 44100]
NFFTS = [512, 1024, 2048, 4096]
HOP_LENGTHS = [128, 256, 512, 1024]
CLIP_DURATION = 5

YOLO_IMAGES_DIR = DATA_PREPROCESSED_DIR / "yolo_images"
YOLO_LABELS_DIR = DATA_PREPROCESSED_DIR / "yolo_labels"

YOLO_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
YOLO_LABELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for sr in SAMPLE_RATES:
    for nfft in NFFTS:
        for hop in HOP_LENGTHS:
            waveform, _ = librosa.load(sample_record, sr=sr)
            raw_annotations: list[BBoxAnnotation] = []
            for _, row in annotations.iterrows():
                raw_annotations.append(
                    BBoxAnnotation(
                        specie=row["specie"],
                        call_type=row["call_type"],
                        begin_time=float(row["begin_time"]),
                        end_time=float(row["end_time"]),
                        low_freq=float(row["low_freq"]),
                        high_freq=float(row["high_freq"]),
                    )
                )
            if raw_annotations:
                target_event = random.choice(raw_annotations)

                clip_audio, clip_annotations = extract_anchored_clip(
                    waveform=waveform,
                    annotations=raw_annotations,
                    target_event=target_event,
                    sample_rate=sr,
                    clip_duration=CLIP_DURATION,
                )
            else:
                max_start = max(0, len(waveform) - int(CLIP_DURATION * sr))
                random_start_sample = random.randint(0, max_start)
                offset_sec = random_start_sample / sr

                clip_audio = extract_clip(waveform, sr, offset_sec, CLIP_DURATION)
                clip_annotations = []

            rgb_spectrogram = audio_to_rgb_spectrogram(
                audio_clip=clip_audio, nfft=nfft, hop_length=hop
            )
            coords = annotations_to_yolo_coords(
                clip_annotations,
                CLIP_DURATION,
                sr,
                lambda specie, call_type: get_species_id(specie),
            )

            base_filename = f"{sample_record.stem}_clip"
            image_filename = f"{base_filename}_sr{sr}_nfft{nfft}_hop{hop}.png"
            label_filename = f"{base_filename}_sr{sr}_nfft{nfft}_hop{hop}.txt"

            cv2.imwrite(str(YOLO_IMAGES_DIR / image_filename), rgb_spectrogram)

            with open(YOLO_LABELS_DIR / label_filename, "w") as f:
                for coord in coords:
                    line = f"{coord['class_id']} {coord['xc_rel']:.6f} {coord['yc_rel']:.6f} {coord['w_rel']:.6f} {coord['h_rel']:.6f}\n"
                    f.write(line)